# Lecture 4 Lab: Benchmarking ANN Indexes with FAISS
**Vector Databases Course** | Estimated time: 60-75 minutes | Platform: Google Colab (CPU is enough)

**Goal:** build four FAISS indexes (Flat, HNSW, IVF-Flat, IVF-PQ) over the same 100,000 vectors and compare them on recall@10, speed and size. Flat search is exact, so it provides the ground truth for measuring recall.

- Measure how exact search cost grows with the dataset.
- Tune `efSearch` (HNSW) and `nprobe` (IVF) and observe the recall/speed trade-off.
- Estimate the memory savings of product quantization.
- Measure memory and tail latency (p95), and choose an index for a given workload.

Run each cell in order. Write your answers in the **YOUR TURN** answer cells.

## Setup
### Cell 1: install and import

In [ ]:
!pip install -q faiss-cpu numpy pandas matplotlib
import os, time
import numpy as np, pandas as pd
import faiss
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)

## Part 1: Data
We generate clustered synthetic vectors, which behave more like real embeddings than uniform random noise. (You can later replace them with sentence embeddings from Lecture 2.)

### Cell 2: dataset

In [ ]:
N, NQ, D, K = 100_000, 1_000, 128, 10
centers = rng.normal(size=(200, D)).astype("float32") * 3
def make(n):
    ids = rng.integers(0, len(centers), n)
    return (centers[ids] + rng.normal(size=(n, D))).astype("float32")
xb = make(N)   # database vectors
xq = make(NQ)  # query vectors
print(xb.shape, xq.shape)

## Part 2: Ground truth and helpers
### Cell 3: Flat index (exact) and benchmark helper

In [ ]:
flat = faiss.IndexFlatL2(D)
flat.add(xb)
_, I_true = flat.search(xq, K)   # exact top-10 for every query

def recall_at_k(I):
    return float(np.mean([len(set(a) & set(b)) / K for a, b in zip(I, I_true)]))

def bench(index):
    t = time.perf_counter()
    _, I = index.search(xq, K)
    dt = time.perf_counter() - t
    return recall_at_k(I), NQ / dt, dt / NQ * 1000   # recall, QPS, ms/query

def size_mb(index):
    faiss.write_index(index, "tmp.index")
    return os.path.getsize("tmp.index") / 1e6

r, qps, ms = bench(flat)
print(f"Flat recall={r:.3f} QPS={qps:,.0f} size={size_mb(flat):.1f} MB")

**Expected:** recall is exactly 1.000. Note the QPS and size; they are your baseline.

> **YOUR TURN:** Run Cell 3 twice. Why do QPS values differ slightly between runs? Write one sentence.

*(write here)*

## Part 3: HNSW
### Cell 4: build HNSW and sweep efSearch

In [ ]:
summary = {"Flat": dict(size=size_mb(flat), build=None)}   # bookkeeping for the Part 7 table

t = time.perf_counter()
hnsw = faiss.IndexHNSWFlat(D, 32)   # M = 32 links per node
hnsw.hnsw.efConstruction = 200
hnsw.add(xb)
build_hnsw = time.perf_counter() - t
print(f"HNSW build time: {build_hnsw:.1f} s, size={size_mb(hnsw):.1f} MB")
summary["HNSW"] = dict(size=size_mb(hnsw), build=build_hnsw)

rows = []
for ef in [4, 8, 16, 32, 64, 128, 256]:
    hnsw.hnsw.efSearch = ef
    r, qps, ms = bench(hnsw)
    rows.append(dict(index="HNSW", param=f"efSearch={ef}", recall=r, qps=qps, ms=ms))
pd.DataFrame(rows)

## Part 4: IVF-Flat
### Cell 5: build IVF-Flat and sweep nprobe

In [ ]:
nlist = 1024   # about 3 x sqrt(N)
quantizer = faiss.IndexFlatL2(D)
ivf = faiss.IndexIVFFlat(quantizer, D, nlist)
t = time.perf_counter()
ivf.train(xb)   # k-means on the data
ivf.add(xb)
build_ivf = time.perf_counter() - t
print(f"IVF build time: {build_ivf:.1f} s, size={size_mb(ivf):.1f} MB")
summary["IVF-Flat"] = dict(size=size_mb(ivf), build=build_ivf)

for npb in [1, 2, 4, 8, 16, 32, 64]:
    ivf.nprobe = npb
    r, qps, ms = bench(ivf)
    rows.append(dict(index="IVF-Flat", param=f"nprobe={npb}", recall=r, qps=qps, ms=ms))
pd.DataFrame(rows).tail(7)

## Part 5: IVF-PQ
Product quantization stores each 128-d vector as m = 16 bytes instead of 512 bytes.

### Cell 6: build IVF-PQ and sweep nprobe

In [ ]:
m = 16   # sub-vectors; D must be divisible by m
quantizer2 = faiss.IndexFlatL2(D)
ivfpq = faiss.IndexIVFPQ(quantizer2, D, nlist, m, 8)   # 8 bits = 256 centroids per part
t = time.perf_counter()
ivfpq.train(xb)
ivfpq.add(xb)
build_pq = time.perf_counter() - t
print(f"IVF-PQ build time: {build_pq:.1f} s, size={size_mb(ivfpq):.1f} MB")
summary["IVF-PQ"] = dict(size=size_mb(ivfpq), build=build_pq)

for npb in [1, 2, 4, 8, 16, 32, 64]:
    ivfpq.nprobe = npb
    r, qps, ms = bench(ivfpq)
    rows.append(dict(index="IVF-PQ", param=f"nprobe={npb}", recall=r, qps=qps, ms=ms))

## Part 6: Memory arithmetic and tail latency
### Cell 7: bytes per vector and p95 latency

In [ ]:
bytes_flat = D * 4   # float32
print(f"Flat   : {bytes_flat} B/vector")
print(f"IVF-PQ : {m} B/vector codes, {bytes_flat / m:.0f}x smaller")
print(f"HNSW   : +{2 * 32 * 4} B/vector of links at M=32")

hnsw.hnsw.efSearch = 64
lat = []
for q in xq[:300]:
    t0 = time.perf_counter()
    hnsw.search(q[None, :], K)
    lat.append((time.perf_counter() - t0) * 1000)
p50, p95, p99 = np.percentile(lat, [50, 95, 99])
print(f"p50 {p50:.2f} ms | p95 {p95:.2f} ms | p99 {p99:.2f} ms")

In [ ]:
# Helper for the YOUR TURN below: bytes per vector implied by the saved index files
for name, s in summary.items():
    print(f"{name:9s} {s['size']:7.1f} MB  ->  {s['size'] * 1e6 / N:7.1f} B/vector")

> **YOUR TURN:** Why can p95 be much larger than the average, and why does it matter for a user-facing search box? Compare the Flat and HNSW bytes per vector with the numbers from the saved index files.

*(write here)*

## Part 7: Compare
### Cell 8: recall vs speed curve

In [ ]:
df = pd.DataFrame(rows)
for name, g in df.groupby("index"):
    plt.plot(g["recall"], g["qps"], marker="o", label=name)
plt.yscale("log"); plt.xlabel("recall@10"); plt.ylabel("queries per second (log)")
plt.legend(); plt.grid(alpha=0.3); plt.title("Recall vs speed"); plt.show()
df.round(3)

> **YOUR TURN:** Record your measurements in the table below (pick the setting that first reaches recall of at least 0.90; write 'none' if the index never does). The next cell fills it in automatically; copy it into the markdown table or paste your own values.

In [ ]:
# Auto-fill: first setting per index with recall >= 0.90
_, flat_qps, _ = bench(flat)
table = [dict(Index="Flat", Setting="exact", QPS=round(flat_qps),
              Size_MB=round(summary["Flat"]["size"], 1), Build_s="-")]
for name in ["HNSW", "IVF-Flat", "IVF-PQ"]:
    g = df[(df["index"] == name) & (df["recall"] >= 0.90)]
    s = summary[name]
    if len(g):
        first = g.iloc[0]   # sweeps are in increasing-effort order
        table.append(dict(Index=name, Setting=first["param"], QPS=round(first["qps"]),
                          Size_MB=round(s["size"], 1), Build_s=round(s["build"], 1)))
    else:
        table.append(dict(Index=name, Setting="none", QPS="-",
                          Size_MB=round(s["size"], 1), Build_s=round(s["build"], 1)))
pd.DataFrame(table)

| Index | Setting for recall >= 0.90 | QPS | Size (MB) | Build time (s) |
|---|---|---|---|---|
| Flat | | | | |
| HNSW | | | | |
| IVF-Flat | | | | |
| IVF-PQ | | | | |

## Part 8: Explore
- Set M to 8, then 64 for HNSW. How do build time, size and recall change?
- Try nlist = 256 and nlist = 4096 for IVF. Which needs a larger nprobe for the same recall?
- Increase N to 500,000 and re-measure Flat. Does the cost grow linearly, as the lecture predicts?
- Use inner product (`IndexFlatIP`) on normalized sentence embeddings instead of synthetic data.

In [ ]:
# 8a: HNSW with different M (efSearch fixed at 64)
for M in [8, 64]:
    t = time.perf_counter()
    h = faiss.IndexHNSWFlat(D, M)
    h.hnsw.efConstruction = 200
    h.add(xb)
    bt = time.perf_counter() - t
    h.hnsw.efSearch = 64
    r, qps, ms = bench(h)
    print(f"M={M:3d}: build {bt:5.1f} s | size {size_mb(h):6.1f} MB | recall {r:.3f} | QPS {qps:,.0f}")

In [ ]:
# 8b: IVF-Flat with different nlist
for nl in [256, 4096]:
    qz = faiss.IndexFlatL2(D)
    ix = faiss.IndexIVFFlat(qz, D, nl)
    ix.train(xb); ix.add(xb)
    for npb in [1, 2, 4, 8, 16, 32, 64]:
        ix.nprobe = npb
        r, qps, ms = bench(ix)
        print(f"nlist={nl:5d} nprobe={npb:3d}: recall {r:.3f} | QPS {qps:,.0f}")
        if r >= 0.90:
            break

In [ ]:
# 8c: Flat search cost at larger N (needs ~256 MB for 500k x 128 float32 vectors)
N2 = 500_000
xb2 = make(N2)
flat2 = faiss.IndexFlatL2(D); flat2.add(xb2)
t = time.perf_counter(); flat2.search(xq, K); t2 = time.perf_counter() - t
t = time.perf_counter(); flat.search(xq, K);  t1 = time.perf_counter() - t
print(f"N={N:,}: {t1:.2f} s | N={N2:,}: {t2:.2f} s | ratio {t2/t1:.1f}x for {N2/N:.0f}x more data")
del xb2, flat2

**My observations for Part 8:**

*(write here)*

## Reflection questions

**1. Which index gave the best recall at the lowest latency, and what did it cost in memory and build time?**

*(write here)*

**2. Why does IVF-PQ plateau at a recall below 1.0 even with a very large nprobe?**

*(write here)*

**3. You must serve 200 million vectors on a machine with 64 GB of RAM. Which index would you start with, and why?**

*(write here)*

**4. Why is it important to measure recall on your own data rather than trusting a published benchmark?**

*(write here)*

---
## Deliverable
Submit the executed notebook, the recall-vs-speed plot, the completed table, and short answers to the reflection questions.

## Troubleshooting
- **Kernel runs out of memory:** reduce N to 50,000.
- **Absolute QPS differs from classmates:** Colab hardware varies; compare indexes against each other, not against other machines.
- **AssertionError on IVFPQ:** D must be divisible by m.